# 03. Diagnostics: Cause, Symptom, and Competing Explanations

Phase 3, 2026-08-21. Answers Questions 3 through 7 in `docs/locked_business_questions.md`.

**Pre-registered primary comparison, stated before any other cut is run:** Enterprise win rate, pre-push vs. post-push, cohorted by `opened_date`. Every other comparison in this notebook is secondary and exploratory, and is labelled as such.

In [1]:
import sys
sys.path.insert(0, "../src")
import phase3_lib as lib
import pandas as pd
pd.set_option("display.width", 160)
pd.set_option("display.max_rows", 100)

tables = lib.load_all()
deals, reps, stage_history, pipeline, spend = (
    tables["deals"], tables["sales_reps"], tables["deal_stage_history"],
    tables["sales_pipeline"], tables["sm_spend"]
)

## Primary comparison (pre-registered)

Enterprise win rate, pre-push vs. post-push, cohorted by `opened_date`.

In [2]:
ent = deals[deals["segment"] == "Enterprise"]
pre = ent[ent["opened_date"] < lib.ENTERPRISE_PUSH_DATE]
post = ent[ent["opened_date"] >= lib.ENTERPRISE_PUSH_DATE]
primary = lib.two_prop_ci(pre["deal_won"].sum(), len(pre), post["deal_won"].sum(), len(post))
print(f"Pre-push:  {primary['p1']:.1%}  (n={primary['n1']})")
print(f"Post-push: {primary['p2']:.1%}  (n={primary['n2']})")
print(f"Difference: {primary['diff']*100:.1f}pp, 95% CI [{primary['ci_lo']*100:.1f}, {primary['ci_hi']*100:.1f}]pp, z={primary['z']:.2f}")
print()
print("CI excludes zero:", primary["ci_lo"] > 0)

Pre-push:  35.2%  (n=105)
Post-push: 10.2%  (n=315)
Difference: 25.1pp, 95% CI [15.4, 34.8]pp, z=5.05

CI excludes zero: True


**This is the headline finding.** The confidence interval clears zero comfortably at this sample size (n=420 total Enterprise attempts, up from 150 in the original plan specifically to fix this). Everything below investigates *why*, not *whether*.

## Question 3: cause vs. symptom, close-date vs. open-date cohorting

In [3]:
close_q = lib.win_rate_by_quarter(deals, "close_date", "Enterprise")
open_q = lib.win_rate_by_quarter(deals, "opened_date", "Enterprise")
print("By CLOSE date:")
print(close_q.to_string(index=False))
print()
print("By OPEN date:")
print(open_q.to_string(index=False))

By CLOSE date:
quarter  win_rate  n
 2024Q3  0.444444 36
 2024Q4  0.257143 35
 2025Q1  0.217391 46
 2025Q2  0.071429 42
 2025Q3  0.078947 38
 2025Q4  0.095238 63
 2026Q1  0.129032 62
 2026Q2  0.160714 56
 2026Q3  0.119048 42

By OPEN date:
quarter  win_rate  n
 2024Q1  0.750000  4
 2024Q2  0.521739 23
 2024Q3  0.354839 31
 2024Q4  0.234043 47
 2025Q1  0.085106 47
 2025Q2  0.092593 54
 2025Q3  0.137255 51
 2025Q4  0.156250 64
 2026Q1  0.040816 49
 2026Q2  0.111111 36
 2026Q3  0.000000 14


In [4]:
# Lag quantification, two angles.
# 1. Empirical break point: first quarter each cohorting method shows a
#    sustained, clear drop from the prior baseline.
print("By open date, the break is exactly where the mechanism places it: 2025 Q1,")
print("the push date, since that split defines the cohort boundary by construction.")
print()
print("By close date, win rate holds near baseline through 2025 Q1 (21.7%) and drops")
print("sharply in 2025 Q2 (7.1%). That is a one-quarter (roughly 3-month) reporting lag")
print("between when the cause started and when a close-date-only view would have noticed.")
print()

# 2. Mechanistic angle: how long a real (non-instant-reject) post-push deal
#    actually takes to resolve, which is *why* the lag exists.
post = deals[(deals["segment"] == "Enterprise") & (deals["opened_date"] >= lib.ENTERPRISE_PUSH_DATE)]
real_outcome = post[post["loss_reason"] != "unqualified"]
instant_reject = post[post["loss_reason"] == "unqualified"]
print(f"Post-push deals that reach a real outcome (stalled/lost/won): mean cycle "
      f"{real_outcome['sales_cycle_days'].mean()/30.44:.1f} months, n={len(real_outcome)}")
print(f"Post-push deals rejected at qualification: mean cycle "
      f"{instant_reject['sales_cycle_days'].mean()/30.44:.1f} months, n={len(instant_reject)}")

By open date, the break is exactly where the mechanism places it: 2025 Q1,
the push date, since that split defines the cohort boundary by construction.

By close date, win rate holds near baseline through 2025 Q1 (21.7%) and drops
sharply in 2025 Q2 (7.1%). That is a one-quarter (roughly 3-month) reporting lag
between when the cause started and when a close-date-only view would have noticed.

Post-push deals that reach a real outcome (stalled/lost/won): mean cycle 5.9 months, n=117
Post-push deals rejected at qualification: mean cycle 0.6 months, n=198


**Reading this.** Two different lag figures, and they answer two different questions. The **reporting lag** (about one quarter, ~3 months) is how long a normal monthly/quarterly business review watching win rate by close date would take to notice something changed, since the founder-led deals still closing in early 2025 mask the new team's performance until the new team's own deals start resolving. The **mechanistic cycle length** (~5.9 months for deals that reach a real decision) is longer than that, because most of the *volume* of post-push deals are quick unqualified rejections (mean 0.6 months) that resolve fast and pull the blended average down; it is specifically the deals that seriously progress toward Proposal and Negotiation that take the longest and are the ones the stall mechanism affects. Both numbers matter for different purposes: the first tells you how much of a head start a bad decision gets before the close-date dashboard would flag it, the second explains why.

## Question 4: where deals fail, and the mix of loss reasons

In [5]:
stage_df = lib.stage_duration_table(stage_history, deals, "Enterprise", lib.enterprise_period_label)
stage_df

,period,stage,mean,count
2,post-push,Prospecting,17.565079,315
3,post-push,Qualified,25.982906,117
1,post-push,Proposal,57.470085,117
0,post-push,Negotiation,78.675214,117
6,pre-push,Prospecting,15.142857,105
7,pre-push,Qualified,17.415385,65
5,pre-push,Proposal,34.400000,65
4,pre-push,Negotiation,36.476923,65


**Stage duration, pre-push vs. post-push, mean days.** Prospecting and Qualified move modestly; Proposal and Negotiation blow out. This is the stage concentration Question 4 asks for: if the increase were even across all four stages, or concentrated in Prospecting/Qualified instead, that would point away from the stakeholder-engagement mechanism specifically.

In [6]:
ct, counts = lib.loss_reason_mix(deals, "Enterprise", lib.enterprise_period_label)
print("Loss reason mix, share of losses:")
print(ct)
print()
print("Loss reason mix, counts:")
print(counts)

Loss reason mix, share of losses:
loss_reason  lost_to_competitor  stalled_no_decision  unqualified
period                                                           
post-push                 0.099                0.201        0.700
pre-push                  0.412                0.000        0.588

Loss reason mix, counts:
loss_reason  lost_to_competitor  stalled_no_decision  unqualified
period                                                           
post-push                    28                   57          198
pre-push                     28                    0           40


**This is the clearest single result in the dataset.** `stalled_no_decision` is 0% of pre-push Enterprise losses and 20.1% of post-push losses (57 deals). `lost_to_competitor`'s share of losses actually *fell*, from 41.2% to 9.9%. **This directly addresses the Question 4 falsification condition**: it would have been falsified by losses concentrating in `lost_to_competitor` (a positioning/pricing problem) rather than `stalled_no_decision` (an execution problem). The data shows the opposite of that falsifying pattern, and a loss reason that did not exist before the push now accounts for a fifth of all post-push Enterprise losses.

## Question 5: does rep capability explain it

**Confounding note, read before the numbers.** All 8 new Enterprise AEs share a hire date (`ENTERPRISE_PUSH_DATE`), so `has_enterprise_experience` is perfectly confounded with calendar period for that group. The contemporaneous control amendment (`docs/decisions.md`, 2026-08-21) has the 2 founder-led reps continue carrying roughly 12% of post-push Enterprise deals, which is what makes the second comparison below possible.

In [7]:
rc = lib.rep_experience_comparison(deals, reps)

full = rc["full_confounded"]
print("Full comparison (confounded with calendar period):")
print(f"  Experienced:   {full['p1']:.1%} (n={full['n1']})")
print(f"  Inexperienced: {full['p2']:.1%} (n={full['n2']})")
print(f"  Diff: {full['diff']*100:.1f}pp, 95% CI [{full['ci_lo']*100:.1f}, {full['ci_hi']*100:.1f}]pp")
print()

sp = rc["same_period"]
print("Same-period comparison (post-push only, both groups share the same calendar period):")
print(f"  Experienced (founder-led):   {sp['p1']:.1%} (n={sp['n1']})")
print(f"  Inexperienced (new AEs):     {sp['p2']:.1%} (n={sp['n2']})")
print(f"  Diff: {sp['diff']*100:.1f}pp, 95% CI [{sp['ci_lo']*100:.1f}, {sp['ci_hi']*100:.1f}]pp, z={sp['z']:.2f}")
print()
print("Controls (same-period groups, checking they weren't handed systematically different deals):")
for k, v in rc["controls"].items():
    print(f"  {k}: {v}")

Full comparison (confounded with calendar period):
  Experienced:   32.2% (n=143)
  Inexperienced: 8.3% (n=277)
  Diff: 23.9pp, 95% CI [15.5, 32.2]pp

Same-period comparison (post-push only, both groups share the same calendar period):
  Experienced (founder-led):   23.7% (n=38)
  Inexperienced (new AEs):     8.3% (n=277)
  Diff: 15.4pp, 95% CI [1.5, 29.3]pp, z=2.17

Controls (same-period groups, checking they weren't handed systematically different deals):
  exp_post_mean_deal_size: 91219.36289473684
  inexp_post_mean_deal_size: 105185.98411552348
  exp_post_region_mix: {'US': 0.711, 'EU': 0.184, 'APAC': 0.105}
  inexp_post_region_mix: {'US': 0.614, 'EU': 0.235, 'APAC': 0.152}


**Reading this.** The full comparison is large (24pp) but cannot be trusted at face value, since every inexperienced-rep deal is also a post-push deal; it cannot separate a rep effect from a time effect. The same-period comparison, restricted to post-push deals only so both groups share the same calendar period, still shows a real gap (15.4pp, 95% CI [1.5, 29.3]pp) that clears zero, though with a much wider interval given the small founder-led post-push sample (n=38). The deal-size control shows inexperienced reps were not handed systematically *easier* (smaller) deals; if anything their average deal was somewhat larger. The region control shows a modest tilt: inexperienced reps carried slightly more EU volume, which, given the EU competitive pressure examined in Question 6, would work *against* them, not for them, meaning the true experience effect if anything is understated here rather than overstated.

**What this does and does not establish.** It supports rep capability as a real, same-period contributor to the decline, at a magnitude of roughly 15 percentage points, distinguishable from zero but with a wide interval at this sample size. It does not, on its own, rule out every other explanation acting alongside it. Question 6 addresses those directly.

## Question 6: quantifying the four competing explanations

Each is addressed with a number, not dismissed in prose. The rep-capability hypothesis is not assumed to be the answer; any of these four could carry real weight.

### A. EU competitive entry (2025-07-01)

Two methods, since the naive version turns out to be misleading here.

In [8]:
c1 = lib.confounder_eu_competitor(deals)
print("Method 1: pre/post-push decline by region (ENTERPRISE_PUSH_DATE split)")
for region, r in c1["by_region"].items():
    print(f"  {region}: {r['diff']*100:.1f}pp decline, 95% CI [{r['ci_lo']*100:.1f}, {r['ci_hi']*100:.1f}]pp, n={r['n1']}/{r['n2']}")
print(f"  Non-EU baseline decline: {c1['non_eu_baseline_decline_pp']*100:.1f}pp")
print(f"  EU decline: {c1['eu_decline_pp']*100:.1f}pp")
print(f"  EU excess over non-EU baseline: {c1['eu_excess_decline_pp']*100:.1f}pp")
print(f"  Estimated share of overall decline: {c1['estimated_eu_competitor_share_of_overall_decline']*100:.1f}%")

Method 1: pre/post-push decline by region (ENTERPRISE_PUSH_DATE split)
  US: 22.2pp decline, 95% CI [10.4, 33.9]pp, n=72/197
  EU: 24.6pp decline, 95% CI [2.9, 46.3]pp, n=19/72
  APAC: 39.1pp decline, 95% CI [11.4, 66.8]pp, n=14/46
  Non-EU baseline decline: 24.9pp
  EU decline: 24.6pp
  EU excess over non-EU baseline: -0.3pp
  Estimated share of overall decline: -0.3%


In [9]:
# Method 2: this method dilutes the competitor's signal, because it splits
# at the PUSH date (Jan 2025), not the COMPETITOR'S ENTRY date (Jul 2025).
# Many "post-push" EU deals were opened before the competitor arrived. A
# difference-in-differences within the post-push cohort only, split at the
# competitor's actual entry date, isolates the competitor's marginal effect
# from the execution effect (which is constant across the whole post-push
# period by construction of this comparison).
post = deals[(deals["segment"] == "Enterprise") & (deals["opened_date"] >= lib.ENTERPRISE_PUSH_DATE)]
eu, non_eu = post[post["region"] == "EU"], post[post["region"] != "EU"]

before_eu = eu[eu["opened_date"] < lib.COMPETITOR_EU_ENTRY_DATE]
after_eu = eu[eu["opened_date"] >= lib.COMPETITOR_EU_ENTRY_DATE]
before_non = non_eu[non_eu["opened_date"] < lib.COMPETITOR_EU_ENTRY_DATE]
after_non = non_eu[non_eu["opened_date"] >= lib.COMPETITOR_EU_ENTRY_DATE]

p_eu_before, p_eu_after = before_eu["deal_won"].mean(), after_eu["deal_won"].mean()
p_non_before, p_non_after = before_non["deal_won"].mean(), after_non["deal_won"].mean()
did = (p_eu_after - p_eu_before) - (p_non_after - p_non_before)

print("Method 2: difference-in-differences, within post-push cohort only, split at COMPETITOR_EU_ENTRY_DATE")
print(f"  EU:     {p_eu_before:.1%} -> {p_eu_after:.1%}  (n={len(before_eu)}/{len(after_eu)})")
print(f"  Non-EU: {p_non_before:.1%} -> {p_non_after:.1%}  (n={len(before_non)}/{len(after_non)})")
print(f"  DiD estimate (EU-specific incremental decline after competitor entry): {did*100:.1f}pp")

Method 2: difference-in-differences, within post-push cohort only, split at COMPETITOR_EU_ENTRY_DATE
  EU:     10.0% -> 4.8%  (n=30/42)
  Non-EU: 8.5% -> 12.2%  (n=71/172)
  DiD estimate (EU-specific incremental decline after competitor entry): -9.0pp


**Reading this.** Method 1 (naive pre/post-push split by region) finds essentially no EU-specific excess decline, because it dilutes the comparison: many "post-push" EU deals were opened before the competitor actually arrived in July 2025. Method 2 isolates the window that matters, restricting to the post-push cohort only (holding the execution problem constant across both groups) and splitting at the competitor's actual entry date. That shows EU win rate falling further (10.0% to 4.8%) while non-EU rises slightly (8.5% to 12.2%) over the same window, a difference-in-differences of roughly -9pp. At this sample size (n=30 to 172 per cell) that estimate does not reach conventional statistical significance and should be read as suggestive, not conclusive. **Conclusion: the EU competitor most likely carries a small, real, but not statistically confirmable contribution to the Enterprise decline, on the order of single-digit percentage points of the roughly 25pp overall gap, not the primary driver.** This is consistent with a "partially real" confounder, and the honest report is the range and its uncertainty, not a single confident number.

### B. List price increase (2025-08-01, +12%)

In [10]:
c2 = lib.confounder_price_increase(deals)
for label, r in c2.items():
    print(f"{label}: {r['diff']*100:+.1f}pp change after the increase, 95% CI [{r['ci_lo']*100:.1f}, {r['ci_hi']*100:.1f}]pp, n={r['n1']}/{r['n2']}")

Mid-Market: +2.7pp change after the increase, 95% CI [-3.6, 9.1]pp, n=418/342
SMB: +6.6pp change after the increase, 95% CI [0.8, 12.3]pp, n=550/470
Enterprise_post_push_only: -0.4pp change after the increase, 95% CI [-7.2, 6.4]pp, n=121/194


**Reading this.** Mid-Market and SMB, which absorbed the identical 12% increase with no rep-experience issue, show *no* significant negative movement, if anything a small positive one for SMB. Enterprise, restricted to the post-push period only (holding rep experience constant), shows essentially zero change (-0.4pp, CI comfortably straddling zero) across the price-increase date. **Conclusion: price is ruled out as a material driver of the Enterprise decline.**

### C. Seasonality (Q4 peak, Q1 trough)

In [11]:
c3 = lib.confounder_seasonality(deals)
for seg in lib.SEGMENTS:
    print(f"\n{seg}:")
    for q, vals in c3[seg].items():
        print(f"  {q}: {vals['mean']:.1%} (n={vals['count']})")


Enterprise:
  2024Q3: 44.4% (n=36)
  2024Q4: 25.7% (n=35)
  2025Q1: 21.7% (n=46)
  2025Q2: 7.1% (n=42)
  2025Q3: 7.9% (n=38)
  2025Q4: 9.5% (n=63)
  2026Q1: 12.9% (n=62)
  2026Q2: 16.1% (n=56)
  2026Q3: 11.9% (n=42)

Mid-Market:
  2024Q3: 20.4% (n=98)
  2024Q4: 24.7% (n=85)
  2025Q1: 29.9% (n=77)
  2025Q2: 29.3% (n=82)
  2025Q3: 29.5% (n=95)
  2025Q4: 24.5% (n=98)
  2026Q1: 25.3% (n=75)
  2026Q2: 32.3% (n=93)
  2026Q3: 31.6% (n=57)

SMB:
  2024Q3: 33.3% (n=120)
  2024Q4: 38.5% (n=143)
  2025Q1: 38.0% (n=121)
  2025Q2: 28.3% (n=99)
  2025Q3: 38.4% (n=112)
  2025Q4: 32.2% (n=121)
  2026Q1: 20.3% (n=118)
  2026Q2: 35.5% (n=107)
  2026Q3: 34.2% (n=79)


**Reading this.** A purely seasonal effect should show up as a recurring pattern that reverts: the same quarter should look similar year over year, and other segments facing the same calendar should show a comparable wobble. Neither holds here. Enterprise's decline is a sustained structural break that persists across five consecutive quarters (2025 Q2 through 2026 Q2) without reverting toward the pre-push baseline, and neither Mid-Market nor SMB shows an analogous sustained decline over the same calendar window; both fluctuate in a band without a persistent downward trend. **Conclusion: seasonality is ruled out.**

### D. Partnerships channel mix

In [12]:
c4 = lib.confounder_channel_mix(deals)
print("Enterprise channel share, pre-push:", c4["channel_share_pre"])
print("Enterprise channel share, post-push:", c4["channel_share_post"])
print()
d = c4["direct_sales_only_pre_post"]
a = c4["all_channels_pre_post"]
print(f"Direct-Sales-only decline: {d['diff']*100:.1f}pp, 95% CI [{d['ci_lo']*100:.1f}, {d['ci_hi']*100:.1f}]pp, n={d['n1']}/{d['n2']}")
print(f"All-channels decline:      {a['diff']*100:.1f}pp, 95% CI [{a['ci_lo']*100:.1f}, {a['ci_hi']*100:.1f}]pp, n={a['n1']}/{a['n2']}")

Enterprise channel share, pre-push: {'Direct Sales': 0.876, 'Inbound': 0.095, 'Partnerships': 0.029}
Enterprise channel share, post-push: {'Direct Sales': 0.886, 'Partnerships': 0.07, 'Inbound': 0.044}

Direct-Sales-only decline: 24.0pp, 95% CI [13.6, 34.4]pp, n=92/279
All-channels decline:      25.1pp, 95% CI [15.4, 34.8]pp, n=105/315


**Reading this.** Enterprise's channel mix barely moves (Direct Sales 87.6% to 88.6% of volume), and restricting the pre/post comparison to Direct Sales only, which removes any channel-mix effect by construction, reproduces virtually the same decline (24.0pp) as the all-channels comparison (25.1pp). **Conclusion: channel mix is ruled out.**

### Summary: apportioning the ~25pp Enterprise decline

| Explanation | Verdict | Estimated contribution |
|---|---|---|
| Rep capability / execution (same-period test) | Real, primary | ~15pp of the same-period comparison clears zero; consistent with being the dominant driver |
| EU competitive entry | Partially real, minor | Single digits, not statistically confirmable at this n; directionally present in the DiD test |
| List price increase | Ruled out | Not distinguishable from zero in any segment |
| Seasonality | Ruled out | No reversion, no analogous pattern in other segments |
| Partnerships channel mix | Ruled out | Decline unchanged when channel mix is held constant |

These do not sum to exactly 25pp; rep capability and the EU competitor are not fully additive or independent (some EU deals are also worked by inexperienced reps), and the same-period rep-capability estimate (15pp) itself carries a wide interval. The honest statement is: **execution is the dominant, statistically supported driver; the EU competitor plausibly adds a small amount on top; the other two are not material.**

## Question 7: why the dashboard missed it, and what it costs

In [13]:
agg_view = lib.aggregate_vs_segment_view(deals)
agg_view

,quarter,blended_win_rate,blended_n,enterprise_win_rate,enterprise_n
0,2024Q3,0.299213,254,0.444444,36
1,2024Q4,0.323194,263,0.257143,35
2,2025Q1,0.323770,244,0.217391,46
3,2025Q2,0.246637,223,0.071429,42
4,2025Q3,0.302041,245,0.078947,38
5,2025Q4,0.244681,282,0.095238,63
6,2026Q1,0.200000,255,0.129032,62
7,2026Q2,0.300781,256,0.160714,56
8,2026Q3,0.280899,178,0.119048,42


**The aggregate (blended, all-segment) win rate stays in a 20-32% band throughout the window and never signals a problem.** Enterprise alone falls from the 22-44% range to the 7-16% range over the same period and stays there. This is Simpson's paradox in a real business metric: a board glancing at the blended number sees nothing wrong.

In [14]:
budget = lib.enterprise_budget_share(spend, deals)
budget

,segment,spend_share,won_deal_share,new_arr_share
0,Enterprise,0.516129,0.107407,0.342794
1,Mid-Market,0.290323,0.374074,0.442757
2,SMB,0.193548,0.518519,0.214449


**Enterprise absorbs 51.6% of the trailing-12-month S&M budget, produces 10.7% of won deals, and 34.3% of new ARR in that period.** More than half the sales and marketing budget for close to a third of new revenue and about a tenth of the deals.

In [15]:
pipe = lib.pipeline_stuck_analysis(pipeline)
pipe

,segment,open_opportunities,open_forecast_usd,late_stage_opportunities,stuck_90d_plus,stuck_share_of_late_stage,stuck_forecast_usd
0,Enterprise,80,8237120.36,57,22,0.385965,2231846.91
1,Mid-Market,126,4079188.67,65,0,0.000000,0.00
2,SMB,194,2088681.94,97,2,0.020619,24534.33


**Forward-looking evidence of the same problem, before it lands in closed-lost.** 38.6% of Enterprise late-stage (Proposal/Negotiation) open pipeline has been sitting 90+ days in stage, representing $2.23M of forecast value, against 0% for Mid-Market and 2.1% for SMB. A CRM pipeline-coverage view (open pipeline value against quota) would show Enterprise pipeline as present and forecastable; `win_probability_pct` follows the same fixed stage convention for every opportunity regardless of how long it has actually been stuck, so the dashboard field that is supposed to flag risk does not distinguish a healthy Proposal-stage deal from one that has been stalled for three months. Only `days_in_current_stage`, a field most pipeline reviews do not surface prominently, reveals it.

## Summary across all seven questions

In [16]:
print("Q3: reporting lag ~1 quarter (close-date vs open-date cohorting); mechanistic cycle for deals that")
print("    reach a real decision ~5.9 months.")
print("Q4: stall concentrated in Proposal/Negotiation; stalled_no_decision share of losses 0% -> 20.1%,")
print("    lost_to_competitor share 41.2% -> 9.9% (falls, does not rise).")
print("Q5: same-period rep-experience gap 15.4pp, 95% CI [1.5, 29.3]pp, n=38/277.")
print("Q6: execution is the dominant driver; EU competitor small and not statistically confirmable;")
print("    price, seasonality, and channel mix ruled out.")
print("Q7: blended win rate never leaves a 20-32% band; Enterprise gets 51.6% of S&M spend for 10.7% of")
print("    won deals and 34.3% of new ARR; 38.6% of late-stage pipeline stuck 90+ days.")

Q3: reporting lag ~1 quarter (close-date vs open-date cohorting); mechanistic cycle for deals that
    reach a real decision ~5.9 months.
Q4: stall concentrated in Proposal/Negotiation; stalled_no_decision share of losses 0% -> 20.1%,
    lost_to_competitor share 41.2% -> 9.9% (falls, does not rise).
Q5: same-period rep-experience gap 15.4pp, 95% CI [1.5, 29.3]pp, n=38/277.
Q6: execution is the dominant driver; EU competitor small and not statistically confirmable;
    price, seasonality, and channel mix ruled out.
Q7: blended win rate never leaves a 20-32% band; Enterprise gets 51.6% of S&M spend for 10.7% of
    won deals and 34.3% of new ARR; 38.6% of late-stage pipeline stuck 90+ days.


## Reproducibility note

This notebook only reads `data/raw/*.csv`; it performs no writes to that directory. Re-running `python src/validate_dataset.py --phase 3` after this notebook confirms the underlying data is untouched.

## Shot list charts

Saved to `notebooks/figures/` for embedding in the Phase 4 case study. See `docs/screenshot_shot_list.md`.

In [17]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import os
os.makedirs("figures", exist_ok=True)

# Brand palette for the chart restyle, locked 2026-08-21 (docs/decisions.md, Decision 24).
# Derived from the live site guide (05_EA_Neyda_AI/brand-guidelines/brand-style-guide.md),
# not the healthcare-ops-finance palette in 04_Brand_and_Portfolio/brand/, which is scoped
# to that project only.
BG = "#FAF8F5"       # site Background
GRID = "#E6E2DC"      # site Line
INK = "#1A1A1A"       # site Ink, titles
MUTED = "#5C5C5C"     # site Muted, axis labels, captions, legend, neutral category
SERIES_A = "#291752"  # site Accent Dark, baseline series (pre-push, blended, close-date)
SERIES_B = "#B5502E"  # new terracotta, contrast series (post-push, Enterprise, open-date)

plt.rcParams.update({
    "figure.facecolor": BG,
    "axes.facecolor": BG,
    "axes.edgecolor": GRID,
    "axes.labelcolor": MUTED,
    "axes.titlecolor": INK,
    "axes.titleweight": "bold",
    "axes.titlesize": 13,
    "axes.labelsize": 10.5,
    "text.color": INK,
    "xtick.color": MUTED,
    "ytick.color": MUTED,
    "xtick.labelsize": 9.5,
    "ytick.labelsize": 9.5,
    "grid.color": GRID,
    "grid.linewidth": 0.8,
    "axes.grid": True,
    "axes.grid.axis": "y",
    "axes.axisbelow": True,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
    "legend.fontsize": 9.5,
    "font.family": "Inter",
})


In [18]:
# Shot 1: aggregate vs Enterprise win rate, quarterly, close date
fig, ax = plt.subplots(figsize=(9, 5))
x = list(agg_view["quarter"].astype(str))
ax.plot(x, agg_view["blended_win_rate"] * 100, marker="o", linestyle="-", label="Blended (all segments)", color=SERIES_A, linewidth=2)
ax.plot(x, agg_view["enterprise_win_rate"] * 100, marker="s", linestyle="--", label="Enterprise only", color=SERIES_B, linewidth=2)
if "2025Q1" in x:
    ax.axvline(x=x.index("2025Q1"), color=MUTED, linestyle=":", alpha=0.7, label="Enterprise push (2025-01-01)")
ax.set_ylabel("Win rate (%)")
ax.set_title("Blended win rate looks healthy; Enterprise alone does not\n(quarterly, cohorted by close date)")
ax.legend()
ax.set_ylim(0, 50)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("figures/01_aggregate_vs_enterprise_win_rate.png", dpi=150)
plt.show()
print("saved figures/01_aggregate_vs_enterprise_win_rate.png")


saved figures/01_aggregate_vs_enterprise_win_rate.png


In [19]:
# Shot 2: Enterprise win rate, open-date vs close-date cohort
close_series = close_q.set_index(close_q["quarter"].astype(str))["win_rate"]
open_series = open_q.set_index(open_q["quarter"].astype(str))["win_rate"]
all_quarters = sorted(set(close_series.index) | set(open_series.index))

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(all_quarters, [close_series.get(q, float("nan")) * 100 for q in all_quarters],
        marker="o", linestyle="-", label="Cohorted by close date", color=SERIES_A, linewidth=2)
ax.plot(all_quarters, [open_series.get(q, float("nan")) * 100 for q in all_quarters],
        marker="s", linestyle="--", label="Cohorted by open date", color=SERIES_B, linewidth=2)
ax.set_ylabel("Enterprise win rate (%)")
ax.set_title("The break appears roughly a quarter earlier when cohorted by open date")
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("figures/02_open_vs_close_date_cohort.png", dpi=150)
plt.show()
print("saved figures/02_open_vs_close_date_cohort.png")


saved figures/02_open_vs_close_date_cohort.png


In [20]:
# Shot 4: stage duration, Proposal and Negotiation, pre vs post
sub = stage_df[stage_df["stage"].isin(["Proposal", "Negotiation"])]
pivot = sub.pivot(index="stage", columns="period", values="mean")[["pre-push", "post-push"]]
pivot = pivot.reindex(["Proposal", "Negotiation"])

fig, ax = plt.subplots(figsize=(7, 5))
pivot.plot(kind="bar", ax=ax, color=[SERIES_A, SERIES_B])
ax.set_ylabel("Mean days in stage")
ax.set_title("Stage duration blows out specifically in Proposal and Negotiation")
ax.legend(title=None)
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("figures/04_stage_duration_proposal_negotiation.png", dpi=150)
plt.show()
print("saved figures/04_stage_duration_proposal_negotiation.png")


saved figures/04_stage_duration_proposal_negotiation.png


In [21]:
# Shot 5: loss reason mix, pre vs post
fig, ax = plt.subplots(figsize=(7, 5))
ct.reindex(["pre-push", "post-push"])[["unqualified", "stalled_no_decision", "lost_to_competitor"]].plot(
    kind="bar", stacked=True, ax=ax, color=[MUTED, SERIES_B, SERIES_A])
ax.set_ylabel("Share of losses")
ax.set_title("Loss reason mix shifts toward stalled deals post-push")
ax.legend(title="Loss reason", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("figures/05_loss_reason_mix.png", dpi=150)
plt.show()
print("saved figures/05_loss_reason_mix.png")


saved figures/05_loss_reason_mix.png
